In [3]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

tensor([[ 1.6369e-02, -2.5385e-01, -9.3413e-03, -2.4106e-04, -7.1003e-02,
          1.5198e-01, -1.4153e-01, -3.0120e-01, -1.3344e-01,  2.1365e-01],
        [-4.7308e-02, -2.7297e-01, -1.6160e-02, -1.2670e-01, -2.5891e-03,
          1.0187e-01, -1.1105e-01, -2.0273e-01, -1.8360e-01,  8.6892e-02]],
       grad_fn=<AddmmBackward0>)

### 1. 自定义块

1. 将输入数据作为其前向传播函数的参数。

2. 通过前向传播函数来生成输出。请注意，输出的形状可能与输入的形状不同。例如，我们上面模型中的第一个全连接的层接收一个20维的输入，但是返回一个维度为256的输出。

3. 计算其输出关于输入的梯度，可通过其反向传播函数进行访问。通常这是自动发生的。

4. 存储和访问前向传播计算所需的参数。

5. 根据需要初始化模型参数。

In [4]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义。
        return self.out(F.relu(self.hidden(X)))

In [5]:
net = MLP()
print(net(X))
net._modules

tensor([[ 0.0785,  0.1099, -0.3115,  0.0764,  0.1743,  0.0187,  0.0178, -0.2155,
         -0.0575,  0.0839],
        [ 0.0444, -0.0142, -0.3575,  0.1303,  0.2159, -0.0088, -0.0281, -0.2872,
          0.0728,  0.0498]], grad_fn=<AddmmBackward0>)


{'hidden': Linear(in_features=20, out_features=256, bias=True),
 'out': Linear(in_features=256, out_features=10, bias=True)}

### 2. 顺序块

In [6]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

In [15]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
print(net(X))
print(net._modules)
net._modules.values()

tensor([[-0.0271, -0.0092, -0.0021,  0.1225, -0.0611, -0.2166,  0.0517,  0.1214,
         -0.0375, -0.0368],
        [-0.0551, -0.0549, -0.0290, -0.0422, -0.0367, -0.1377,  0.0117,  0.0328,
         -0.0699, -0.0666]], grad_fn=<AddmmBackward0>)
{'0': Linear(in_features=20, out_features=256, bias=True), '1': ReLU(), '2': Linear(in_features=256, out_features=10, bias=True)}


dict_values([Linear(in_features=20, out_features=256, bias=True), ReLU(), Linear(in_features=256, out_features=10, bias=True)])

### 3. 在前向传播函数中执行代码

In [8]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [9]:
net = FixedHiddenMLP()
print(net(X))
net._modules

tensor(-0.1040, grad_fn=<SumBackward0>)


{'linear': Linear(in_features=20, out_features=20, bias=True)}

In [10]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
print(chimera(X))
chimera._modules

tensor(0.2303, grad_fn=<SumBackward0>)


{'0': NestMLP(
   (net): Sequential(
     (0): Linear(in_features=20, out_features=64, bias=True)
     (1): ReLU()
     (2): Linear(in_features=64, out_features=32, bias=True)
     (3): ReLU()
   )
   (linear): Linear(in_features=32, out_features=16, bias=True)
 ),
 '1': Linear(in_features=16, out_features=20, bias=True),
 '2': FixedHiddenMLP(
   (linear): Linear(in_features=20, out_features=20, bias=True)
 )}

### 练习

In [11]:
class ParallelBlock(nn.Module):
    def __init__(self, net1, net2):
        super().__init__()
        self.net1 = net1
        self.net2 = net2
    
    def forward(self, X):
        return torch.cat((self.net1(X), self.net2(X)), dim=1)
        

In [12]:
net = ParallelBlock(nn.Linear(20, 10), nn.Linear(20, 10))
net(X)


tensor([[-1.4894e-01,  5.8919e-05,  1.8718e-01, -1.0026e-01,  3.4683e-02,
         -2.0014e-01,  8.6875e-02,  1.4327e-01,  8.0834e-02,  5.5670e-01,
         -5.6098e-01,  3.5971e-01,  3.3381e-01,  6.1652e-01, -5.3503e-03,
          1.1596e-01,  2.6881e-01, -1.3901e-02, -4.2320e-01, -9.3579e-02],
        [-1.8941e-01, -3.0799e-01,  3.8329e-01,  1.2160e-01, -3.8451e-02,
         -3.1965e-01, -2.2811e-01, -3.6738e-02,  3.0409e-01,  9.0588e-01,
         -4.1447e-01,  6.3526e-01,  3.6897e-01,  3.3959e-01,  1.6077e-01,
         -2.9653e-02, -2.4101e-02, -2.0473e-01, -6.3837e-03, -5.0296e-01]],
       grad_fn=<CatBackward0>)

In [13]:
def make_network(block_class, n, *args):
    layers = [block_class(*args) for _ in range(n)]
    return nn.Sequential(*layers)

In [14]:
net = make_network(nn.Linear, 3, 20, 20)
net(X)

tensor([[-0.2092, -0.0781,  0.0181, -0.1326, -0.0071, -0.0397,  0.1111,  0.1327,
          0.1642, -0.0581, -0.1534, -0.0109,  0.1034,  0.1592,  0.2775, -0.1304,
         -0.0005,  0.0399,  0.0140,  0.0572],
        [-0.1856, -0.1020, -0.0334, -0.1701, -0.1426, -0.1276,  0.1173,  0.1901,
          0.1675, -0.0468, -0.1434,  0.0136,  0.2463,  0.1594,  0.1675, -0.0229,
         -0.0238,  0.1270, -0.0025,  0.0098]], grad_fn=<AddmmBackward0>)